# 04 waits over the day

This notebook shows why the stationary approximations err. Two settings are used, both with two blocks and moderate variability: flat demand and $\pi = 2.0$. For each, three plans are simulated on the 1,000-day reference sample:

- the cheapest feasible plan;
- the pick of SIPP-avg under the per-block verdict;
- the pick of the MOL (at $\pi = 2.0$ the MOL accepts no plan).

The notebook records the mean total wait of the patients by arrival time and sets it against two predictions: the stationary wait of each block and the MOL wait of each admission minute.

**Needs:** the main-sample enumerations of `B2-PAR1.0-moderate` and `B2-PAR2.0-moderate`, the first jobs of notebook 01.

**Outputs:** `figures/fig4_mechanism.png` (Figure 4) and `results/mechanism.csv`, with the numbers quoted in the subsection on the waiting times.

**Run time:** about a minute.

In [1]:
# Setup: storage folder, numba and the core module written by notebook 00
import os, sys, shutil, subprocess, time, math
import numpy as np
import pandas as pd

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/My Drive/clinic2026_final'
else:
    BASE = os.environ.get('CLINIC_BASE', os.path.abspath('clinic2026_final'))
try:
    import numba  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numba'], check=True)

CORE = os.path.join(BASE, 'core', 'clinic_core.py')
if not os.path.exists(CORE):
    raise FileNotFoundError('Run notebook 00 first; it writes the core module to ' + CORE)
LOCAL = '/content/clinic_core_local' if IN_COLAB else os.path.join(BASE, 'core')
os.makedirs(LOCAL, exist_ok=True)
if os.path.abspath(LOCAL) != os.path.abspath(os.path.dirname(CORE)):
    shutil.copy(CORE, LOCAL)          # numba caches its compiled code next to the module, so keep it on local disk
sys.path.insert(0, LOCAL)
import clinic_core as cc
P = cc.make_paths(BASE)
os.makedirs(os.path.join(P['results'], 'parts'), exist_ok=True)
print('Project folder:', BASE)
print('numba available:', cc.HAVE_NUMBA)

Mounted at /content/drive
Project folder: /content/drive/My Drive/clinic2026_final
numba available: True


In [2]:
# Mean wait by arrival time for the cheapest feasible plan and the SIPP-avg and MOL picks
rows = []
info = []
for name in ('B2-PAR1.0-moderate', 'B2-PAR2.0-moderate'):
    s = next(x for x in cc.grid() if x.name == name)
    if not os.path.exists(cc.enum_file(P['data'], s, *cc.MAIN)):
        print(f'{name}: enumeration not available yet; skipped')
        continue
    df = cc.load_table(P['data'], s)
    res, scr = cc.evaluate_all_baselines(s, variants=('avg-block', 'mol-daily'))
    plans = {'Cheapest feasible plan': cc.plan_of_row(cc.cheapest_row(df), s)}
    for v, lab in (('avg-block', 'SIPP-avg pick'), ('mol-daily', 'MOL pick')):
        idx = np.nonzero(res[v]['feas'])[0]
        if len(idx):
            plans[lab] = cc.plan_of_row(int(idx[np.lexsort((res[v]['W'][idx], res[v]['M'][idx]))[0]]), s)
    ev = cc.Evaluator(s, *cc.REFERENCE)
    tab = cc.sipp_block_table(s, 'avg')[cc.CUTOFF]
    mol = scr.mol[cc.CUTOFF]
    prof = s.profile()
    for lab, plan in plans.items():
        mid, wait = cc.waits_by_arrival(ev, plan, bin_minutes=10)
        S = ev.staff_matrix(plan)
        combo = [int(((S[:, b] - 1) * tab['weights']).sum()) for b in range(s.n_blocks)]
        mw = mol.patient_waits(S)
        x = ev.evaluate_many([plan])[0]
        info.append({'setting': name, 'plan label': lab, 'plan': cc.staffing_text(plan, s),
                     'M (1,000 days)': x[cc.METRIC_COLS.index('M')], 'SL (1,000 days)': x[cc.METRIC_COLS.index('SL')]})
        for k, m in enumerate(mid):
            blk = min(int(m // (cc.T / s.n_blocks)), s.n_blocks - 1)
            seg = mw[k * 10:(k + 1) * 10]
            rows.append({'setting': name, 'plan label': lab, 'plan': cc.staffing_text(plan, s), 'minute': m,
                         'mean wait': wait[k], 'rate per hour': float(prof.rate(m)) * 60,
                         'stationary wait': float(tab['wait'][blk, combo[blk]]),
                         'MOL wait': float(np.mean(seg)) if len(seg) else np.nan})
mech = pd.DataFrame(rows)
mech.to_csv(os.path.join(P['results'], 'mechanism.csv'), index=False)
pd.DataFrame(info)

,setting,plan label,plan,"M (1,000 days)","SL (1,000 days)"
0,B2-PAR1.0-moderate,Cheapest feasible plan,"(3,2,2,1 | 2,2,3,2)",1557.646593,0.901243
1,B2-PAR1.0-moderate,SIPP-avg pick,"(2,2,3,2 | 2,2,3,2)",1692.420917,0.978089
2,B2-PAR1.0-moderate,MOL pick,"(3,2,2,2 | 2,2,3,2)",1622.610754,0.949150
3,B2-PAR2.0-moderate,Cheapest feasible plan,"(3,2,3,2 | 2,2,3,2)",1682.835213,0.913775
4,B2-PAR2.0-moderate,SIPP-avg pick,"(2,2,3,2 | 3,2,2,2)",1582.303548,0.814491


In [3]:
# Figure 4: mechanism
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
cc.fig_style()
settings_ = list(dict.fromkeys(mech.setting))
colors = {'Cheapest feasible plan': cc.CAT['blue'], 'SIPP-avg pick': cc.CAT['orange'], 'MOL pick': cc.CAT['aqua']}
fig, axes = plt.subplots(2, len(settings_), figsize=(7.2, 5.4), sharex=True, gridspec_kw={'height_ratios': [1, 2.6]},
                         squeeze=False)
for j, name in enumerate(settings_):
    g = mech[mech.setting == name]
    r = g[g['plan label'] == g['plan label'].iloc[0]]
    axes[0, j].plot(r.minute, r['rate per hour'], color=cc.MUTED, linewidth=1.2)
    axes[0, j].set_ylim(0, 13)
    par = name.split('PAR')[1].split('-')[0]
    axes[0, j].set_title(f'$\\pi$ = {par}, two blocks, moderate variability', fontsize=9)
    top = 0.0
    for lab in dict.fromkeys(g['plan label']):
        h = g[g['plan label'] == lab]
        col = colors.get(lab, cc.INK2)
        axes[1, j].plot(h.minute, h['mean wait'], color=col, label=f"{lab} {h.plan.iloc[0]}")
        axes[1, j].step(h.minute, h['stationary wait'], where='mid', color=col, linewidth=0.9, alpha=0.55)
        axes[1, j].plot(h.minute, h['MOL wait'], color=col, linewidth=0.9, linestyle=':')
        top = max(top, float(np.nanmax(h['mean wait'])))
    axes[1, j].axvline(cc.T / 2, color=cc.MUTED, linewidth=0.6)
    axes[1, j].set_ylim(0, 1.9 * top)
    axes[1, j].set_xticks(range(0, 481, 60))
    axes[1, j].set_xlabel('Arrival time (minutes after opening)')
    hs, ls = axes[1, j].get_legend_handles_labels()
    hs += [Line2D([], [], color=cc.INK2, linewidth=0.9, alpha=0.55), Line2D([], [], color=cc.INK2, linewidth=0.9,
                                                                           linestyle=':')]
    ls += ['thin: stationary wait of the block', 'dotted: MOL wait (clipped at the top)']
    if 'MOL pick' not in set(g['plan label']):
        hs.append(Line2D([], [], linestyle='none'))
        ls.append('MOL accepts no plan')
    axes[1, j].legend(hs, ls, loc='upper center', bbox_to_anchor=(0.5, -0.24), fontsize=6.5, ncol=1)
axes[0, 0].set_ylabel('Walk-ins\nper hour')
axes[1, 0].set_ylabel('Mean total wait (minutes)')
fig.tight_layout()
print('Written', cc.save_figure(fig, os.path.join(P['figures'], 'fig4_mechanism.png')))

Written /content/drive/My Drive/clinic2026_final/figures/fig4_mechanism.png


In [4]:
# Numbers quoted in the text: the lone pharmacist of the cheapest flat-demand plan
s = next(x for x in cc.grid() if x.name == 'B2-PAR1.0-moderate')
df = cc.load_table(P['data'], s)
S = cc.Evaluator(s, 1).staff_matrix(cc.plan_of_row(cc.cheapest_row(df), s))
lam = s.profile().block_rates(s.bounds, c=cc.CUTOFF, kind='avg')
mean_pha = cc.STAGE_MEANS[3]
for b in range(s.n_blocks):
    servers = int(S[3, b])
    rho = lam[b] * mean_pha / servers
    if servers == 1:     # exact M/M/1 relaxation time 1 / (mu (1 - sqrt(rho))^2) (Morse, 1955)
        relax = mean_pha / (1 - math.sqrt(rho)) ** 2
        print(f'Block {b + 1}: one pharmacist at utilization {rho:.2f}; M/M/1 relaxation time {relax:.0f} minutes '
              f'({relax / 60:.1f} hours) against a block of {cc.T / s.n_blocks:.0f} minutes')
    else:
        print(f'Block {b + 1}: {servers} pharmacists at utilization {rho:.2f}')
g = mech[(mech.setting == 'B2-PAR1.0-moderate') & (mech['plan label'] == 'Cheapest feasible plan')]
b1 = g[g.minute < cc.T / 2]
print(f'Cheapest flat-demand plan, block 1: stationary mean total wait {b1["stationary wait"].iloc[0]:.1f} min; '
      f'simulated mean wait by 10-minute bin {b1["mean wait"].min():.1f} to {b1["mean wait"].max():.1f} min; '
      f'MOL wait reaches {b1["MOL wait"].max():.1f} min')

Block 1: one pharmacist at utilization 0.80; M/M/1 relaxation time 718 minutes (12.0 hours) against a block of 240 minutes
Block 2: 2 pharmacists at utilization 0.40
Cheapest flat-demand plan, block 1: stationary mean total wait 34.7 min; simulated mean wait by 10-minute bin 2.9 to 21.5 min; MOL wait reaches 34.7 min
